# Xây một luồng streaming bổ sung dữ liệu liên tục vào đồ thị tri thức

In [ ]:
import requests
import pandas as pd

HEADERS = {"User-Agent": "wikidata-fetch"}

def search_entity(name):

    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "format": "json",
        "limit": 1
    }

    r = requests.get(url, params=params, headers=HEADERS).json()

    if len(r["search"]) == 0:
        raise Exception("Entity not found")

    return r["search"][0]["id"], r["search"][0]["label"]


def get_entity(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    r = requests.get(url, headers=HEADERS).json()

    return r["entities"][qid]


def get_label(entity_id):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"

    try:
        r = requests.get(url, headers=HEADERS).json()
        return r["entities"][entity_id]["labels"]["en"]["value"]
    except:
        return entity_id


def extract_relations(entity_name):

    qid, label = search_entity(entity_name)

    entity = get_entity(qid)

    claims = entity["claims"]

    results = []

    for prop in claims:

        relation = get_label(prop)

        for item in claims[prop]:

            mainsnak = item["mainsnak"]

            if "datavalue" not in mainsnak:
                continue

            value = mainsnak["datavalue"]["value"]

            if isinstance(value, dict) and "id" in value:

                tail = get_label(value["id"])

            else:

                tail = str(value)

            time_value = None

            if "qualifiers" in item:

                for q in ["P585","P580","P582"]:

                    if q in item["qualifiers"]:

                        try:

                            dv = item["qualifiers"][q][0].get("datavalue")

                            if dv and "time" in dv["value"]:

                                time_value = dv["value"]["time"]

                        except:
                            pass

            results.append({

                "entity": label,
                "relation": relation,
                "value": tail,
                "time": time_value

            })

    return pd.DataFrame(results)

In [ ]:
df = extract_relations("Elon Musk")

In [ ]:
df.head(100)

,entity,relation,value,time
0,Elon Musk,sex or gender,male,None
1,Elon Musk,Commons category,Elon Musk,None
2,Elon Musk,country of citizenship,South Africa,+1971-00-00T00:00:00Z
3,Elon Musk,country of citizenship,Canada,+1989-00-00T00:00:00Z
4,Elon Musk,country of citizenship,United States,+2002-00-00T00:00:00Z
...,...,...,...,...
95,Elon Musk,AllMovie person ID,p444169,None
96,Elon Musk,P2604,740940,None
97,Elon Musk,P1266,153422,None
98,Elon Musk,P2605,198916,None


In [ ]:
df.to_csv("df_elon.csv")

#Sửa lại để mapping ID chuẩn hơn

In [ ]:
import requests
import pandas as pd
import time

HEADERS = {"User-Agent": "wikidata-research"}

SEARCH_URL = "https://www.wikidata.org/w/api.php"


# =====================================
# 1. tìm entity ID từ tên
# =====================================

def search_entity(name):

    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "format": "json",
        "limit": 1
    }

    r = requests.get(SEARCH_URL, params=params, headers=HEADERS)

    data = r.json()

    if len(data["search"]) == 0:
        raise Exception("Entity not found")

    return data["search"][0]["id"], data["search"][0]["label"]


# =====================================
# 2. lấy dữ liệu entity
# =====================================

def get_entity(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    r = requests.get(url, headers=HEADERS)

    return r.json()["entities"][qid]


# =====================================
# 3. lấy label từ id
# =====================================

def get_label(entity_id):

    try:

        url = f"https://www.wikidata.org/wiki/Special:EntityData/{entity_id}.json"

        r = requests.get(url, headers=HEADERS)

        return r.json()["entities"][entity_id]["labels"]["en"]["value"]

    except:

        return entity_id


# =====================================
# 4. extract relations
# =====================================

def extract_relations(entity_name):

    qid, label = search_entity(entity_name)

    entity = get_entity(qid)

    claims = entity["claims"]

    rows = []

    for prop in claims:

        relation = get_label(prop)

        for item in claims[prop]:

            mainsnak = item["mainsnak"]

            if "datavalue" not in mainsnak:
                continue

            value = mainsnak["datavalue"]["value"]

            if isinstance(value, dict) and "id" in value:
                tail = get_label(value["id"])
            else:
                tail = str(value)

            time_value = None

            if "qualifiers" in item:

                for q in ["P585","P580","P582"]:

                    if q in item["qualifiers"]:

                        try:
                            dv = item["qualifiers"][q][0]["datavalue"]["value"]
                            time_value = dv["time"]
                        except:
                            pass

            rows.append({
                "entity": label,
                "relation": relation,
                "value": tail,
                "time": time_value
            })

            time.sleep(0.1)  # tránh rate-limit

    return pd.DataFrame(rows)


# =====================================
# 5. chạy thử
# =====================================

df = extract_relations("Elon Musk")

print(df.head(20))

       entity                relation  \
0   Elon Musk           sex or gender   
1   Elon Musk        Commons category   
2   Elon Musk  country of citizenship   
3   Elon Musk  country of citizenship   
4   Elon Musk  country of citizenship   
5   Elon Musk               residence   
6   Elon Musk               residence   
7   Elon Musk               residence   
8   Elon Musk               residence   
9   Elon Musk           date of birth   
10  Elon Musk          place of birth   
11  Elon Musk              occupation   
12  Elon Musk              occupation   
13  Elon Musk              occupation   
14  Elon Musk              occupation   
15  Elon Musk              occupation   
16  Elon Musk              occupation   
17  Elon Musk              occupation   
18  Elon Musk              occupation   
19  Elon Musk              occupation   

                                                value                   time  
0                                                male      

In [ ]:
df.to_csv("elon.csv")

#Tải file data về drive

In [ ]:
!apt update
!apt install lbzip2 -y

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,825 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,780 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Pack

In [ ]:
!bash -c 'wget -O- https://dumps.wikimedia.org/wikidatawiki/entities/latest-all.json.bz2 | lbzip2 -dc > "/content/drive/MyDrive/0_Wiki data/latest-all.json"'

--2026-03-09 13:01:56--  https://dumps.wikimedia.org/wikidatawiki/entities/latest-all.json.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100256063849 (93G) [application/octet-stream]
Saving to: ‘STDOUT’

-                     0%[                    ] 764.58M  1.28MB/s    eta 12h 44m